# Medical-domain VLMs — Med-Flamingo, CheXagent, BioViL-T (Kaggle)

**Right dataset:** labeled **chest X-rays** (`data/benchmark` PneumoniaMNIST, ideally 224px).

**Wrong dataset for these models:** Inspiration4 / GeneLab tabular heatmaps from RQ1. Those are not CXRs; CheXagent and BioViL-T are trained for radiology images.

| Model | HF / source | Task here |
|---|---|---|
| CheXagent-2-3b | `StanfordAIMI/CheXagent-2-3b` | Yes/No pneumonia VQA |
| BioViL-T | `microsoft/BiomedVLP-BioViL-T` + `hi-ml-multimodal` | Zero-shot pneumonia similarity |
| Med-Flamingo | `med-flamingo/med-flamingo` | Few-shot VQA (heavy; often skipped on T4) |

Metrics → `results/medical_domain_table.csv`

### Kaggle setup
1. GPU + **Internet ON**
2. Clone public mirror (private `theap06/shrimpy` won't work):
   `!git clone --depth 1 https://github.com/Srinithyaprogram/shrimpy-kaggle.git /kaggle/working/shrimpy`
3. `%cd /kaggle/working/shrimpy` then Run All

In [ ]:
# 0. Setup cwd + deps
import os, pathlib, subprocess, sys

ROOT = pathlib.Path("/kaggle/working/shrimpy")
if not (ROOT / "scripts" / "medical_vlm_helpers.py").exists():
    ROOT = pathlib.Path.cwd()
    if not (ROOT / "scripts" / "medical_vlm_helpers.py").exists():
        if (ROOT.parent / "scripts" / "medical_vlm_helpers.py").exists():
            ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
print("cwd", pathlib.Path.cwd())

pkgs = [
    "transformers>=4.45.0", "accelerate", "bitsandbytes",
    "pillow", "scikit-learn", "einops", "sentencepiece",
    "hi-ml-multimodal",  # BioViL-T image+text engines
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 1. Load CXR benchmark (regenerate at 224 if only tiny 28px thumbs exist)
from pathlib import Path
import json
from PIL import Image

from medical_vlm_helpers import load_medical_benchmark, PNEUMONIA_QUESTION

samples = load_medical_benchmark(Path.cwd())
sizes = [Image.open(s["image_path"]).size for s in samples]
print(f"n={len(samples)} sizes={set(sizes)} pos={sum(s['label'] for s in samples)}")

if sizes and max(max(s) for s in sizes) < 64:
    print("WARN: images are tiny (<64px). Regenerating PneumoniaMNIST at 224...")
    subprocess.check_call([sys.executable, "scripts/generate_benchmark_dataset.py"])
    samples = load_medical_benchmark(Path.cwd())
    sizes = [Image.open(s["image_path"]).size for s in samples]
    print(f"after regen: n={len(samples)} sizes={set(sizes)}")

Path("results").mkdir(exist_ok=True)
DATASET_NAME = "PneumoniaMNIST (CXR)"

In [ ]:
# 2. Shared scoring
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image

from medical_vlm_helpers import (
    parse_yes_no, compute_table_metrics, table_row,
    PNEUMONIA_QUESTION, BIOVIL_POS_PROMPTS, BIOVIL_NEG_PROMPTS,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
rows = []
pred_path = Path("results/medical_domain_predictions.jsonl")
if pred_path.exists():
    pred_path.unlink()

def finalize(model_name, samples, scores, preds=None, status="completed", note=None):
    y = np.array([s["label"] for s in samples], int)
    scores = np.asarray(scores, float)
    preds = (scores >= 0.5).astype(int) if preds is None else np.asarray(preds, int)
    metrics = compute_table_metrics(y, scores, preds)
    kw = {"note": note} if note else {}
    row = table_row(model_name, DATASET_NAME, metrics, len(samples), status, **kw) if note else table_row(model_name, DATASET_NAME, metrics, len(samples), status)
    rows.append(row)
    with pred_path.open("a") as f:
        for s, sc, pr in zip(samples, scores, preds):
            f.write(json.dumps({"model": model_name, "id": s["id"], "label": s["label"], "score": float(sc), "pred": int(pr)}) + "\n")
    print(pd.Series(row).to_string())
    return row

print("device", DEVICE)

## CheXagent-2-3b

In [ ]:
# 3. CheXagent
from transformers import AutoModelForCausalLM, AutoTokenizer

chex_name = "StanfordAIMI/CheXagent-2-3b"
dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32
tokenizer = AutoTokenizer.from_pretrained(chex_name, trust_remote_code=True)
chex = AutoModelForCausalLM.from_pretrained(
    chex_name, device_map="auto" if DEVICE == "cuda" else None, trust_remote_code=True
)
chex = chex.to(dtype).eval()

scores, preds = [], []
prompt = PNEUMONIA_QUESTION
with torch.inference_mode():
    for s in samples:
        paths = [s["image_path"]]
        query = tokenizer.from_list_format([*[{ "image": p } for p in paths], { "text": prompt }])
        conv = [
            {"from": "system", "value": "You are a helpful radiology assistant."},
            {"from": "human", "value": query},
        ]
        input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
        out = chex.generate(
            input_ids.to(chex.device), do_sample=False, num_beams=1,
            max_new_tokens=32, use_cache=True,
        )[0]
        text = tokenizer.decode(out[input_ids.size(1):-1], skip_special_tokens=True)
        pred = parse_yes_no(text)
        if pred is None:
            pred, score = 0, 0.5
        else:
            score = 0.9 if pred == 1 else 0.1
        preds.append(pred); scores.append(score)

finalize("CheXagent-2-3b", samples, scores, preds)
del chex
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## BioViL-T (zero-shot pneumonia)

In [ ]:
# 4. BioViL-T via hi-ml-multimodal
try:
    from health_multimodal.text.utils import BertEncoderType, get_bert_inference
    from health_multimodal.image import ImageInferenceEngine
    from health_multimodal.image.data.transforms import create_chest_xray_transform_for_inference
    from health_multimodal.image.model.pretrained import get_biovil_t_image_encoder
    from health_multimodal.vlp.inference_engine import ImageTextInferenceEngine

    image_inference = ImageInferenceEngine(
        image_model=get_biovil_t_image_encoder(),
        transform=create_chest_xray_transform_for_inference(resize=512, center_crop_size=448),
    )
    engine = ImageTextInferenceEngine(
        image_inference_engine=image_inference,
        text_inference_engine=get_bert_inference(BertEncoderType.BIOVIL_T_BERT),
    )
    engine.to(torch.device(DEVICE))

    scores, preds = [], []
    for s in samples:
        path = Path(s["image_path"])
        pos = float(engine.get_similarity_score_from_raw_data(image_path=path, query_text=BIOVIL_POS_PROMPTS))
        neg = float(engine.get_similarity_score_from_raw_data(image_path=path, query_text=BIOVIL_NEG_PROMPTS))
        # Map pos vs neg similarity to a probability-like score
        score = float(torch.softmax(torch.tensor([neg, pos]), dim=0)[1])
        pred = 1 if pos > neg else 0
        scores.append(score); preds.append(pred)

    finalize("BioViL-T", samples, scores, preds)
    del engine, image_inference
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
except Exception as exc:
    print("BioViL-T failed:", repr(exc))
    rows.append(table_row("BioViL-T", DATASET_NAME, None, 0, status="error", note=str(exc)[:200]))

## Med-Flamingo (optional / heavy)

Needs OpenFlamingo + Llama-7B weights. On a T4 this often OOMs or needs extra setup. We attempt a minimal path and record `skipped` if unavailable.

In [ ]:
# 5. Med-Flamingo — best-effort; skip cleanly if deps/weights missing
RUN_MED_FLAMINGO = False  # set True only if you have Llama-7B + enough GPU RAM

if not RUN_MED_FLAMINGO:
    rows.append(table_row(
        "Med-Flamingo", DATASET_NAME, None, 0, status="skipped",
        note="Set RUN_MED_FLAMINGO=True after installing OpenFlamingo + Llama-7B (see snap-stanford/med-flamingo).",
    ))
    print("Med-Flamingo skipped (default). Enable RUN_MED_FLAMINGO after setup.")
else:
    try:
        # Placeholder: full Med-Flamingo install is environment-specific.
        # See https://github.com/snap-stanford/med-flamingo
        raise NotImplementedError("Wire local Llama-7B path + med-flamingo checkpoint here")
    except Exception as exc:
        rows.append(table_row("Med-Flamingo", DATASET_NAME, None, 0, status="error", note=str(exc)[:200]))

## Results table

In [ ]:
# 6. Save table
import pandas as pd
from pathlib import Path

cols = ["model", "dataset", "accuracy", "precision", "recall", "f1", "auroc", "mcc", "status", "n_samples", "note"]
table = pd.DataFrame(rows)
for c in cols:
    if c not in table.columns:
        table[c] = float("nan")
table = table[cols]
out = Path("results/medical_domain_table.csv")
table.to_csv(out, index=False)
print("Wrote", out.resolve())
display(table)